# Logistic Regression — Solar PV Risk Score (Leak-Free)
### RTU Anomaly Detection — Corrected Training & Evaluation

**What was wrong in v1:**
- `PR_DEV` and `PR_SLOPE` were used as features but also defined the pseudo-labels → model learned the labeling rule, not a real signal
- Synthetic image scores had perfectly separated class means → trivially separable, not representative of real YOLO output
- Train/test split was random → both sets drawn from same synthetic distribution, no real generalisation test

**What this version does differently:**
- Drops `PR_DEV` and `PR_SLOPE` from features
- Neutralises synthetic image scores (uniform 0.25 — no information until real YOLO runs)
- Cross-plant split: trains on Plant 1, tests on Plant 2
- Reports real-rows-only AUC as the headline metric

## 0 — Install Dependencies

In [1]:
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn matplotlib seaborn joblib --quiet
print('Ready.')

Ready.


## 1 — Imports

In [2]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, json
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    average_precision_score, precision_recall_curve,
    brier_score_loss
)
from sklearn.calibration import calibration_curve
print('Imports OK')

Imports OK


## 2 — Configuration

In [3]:
DATASET_PATH = 'training_dataset.csv'   # UPDATE if needed

# ── LEAK-FREE feature set ──────────────────────────────────────────────
# Removed: PR_DEV, PR_SLOPE
# These were computed from the same thresholds used to generate labels.
# Keeping them gives the model perfect information about the labeling rule.
FEATURE_COLS = [
    'PR',               # Raw performance ratio — independent signal
    'TEMP_DELTA',       # Module - ambient temp
    'DC_AC_RATIO',      # Inverter efficiency proxy
    'PR_ROLL_MEAN',     # Smoothed trend — lagged, not circular
    'PR_ROLL_STD',      # Volatility — independent of label rule
    'TEMP_DELTA_SIGMA', # Thermal anomaly z-score
    
]

TARGET_COL = 'label'
FAULT_COL  = 'fault_type'
SYNTH_COL  = 'synthetic'
PLANT_COL  = 'PLANT_ID'   # used for cross-plant split
RANDOM_STATE = 42

print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')

Features (6): ['PR', 'TEMP_DELTA', 'DC_AC_RATIO', 'PR_ROLL_MEAN', 'PR_ROLL_STD', 'TEMP_DELTA_SIGMA']


## 3 — Load Dataset

In [4]:
df = pd.read_csv(DATASET_PATH)

print(f'Rows: {len(df):,}  |  Cols: {df.shape[1]}')
print(f'\nClass distribution:')
for label, count in df[TARGET_COL].value_counts().items():
    name = 'normal' if label == 0 else 'anomaly'
    print(f'  {name}: {count:,} ({count/len(df)*100:.1f}%)')

print(f'\nFault breakdown:')
for ft, count in df[FAULT_COL].value_counts().items():
    print(f'  {ft:12s}: {count:,}')

print(f'\nPlant distribution:')
if PLANT_COL in df.columns:
    for p, count in df[PLANT_COL].value_counts().items():
        print(f'  {p}: {count:,}')
else:
    print(f'  WARNING: column "{PLANT_COL}" not found — will fall back to random split')
    print(f'  Available columns: {list(df.columns)}')

Rows: 88,675  |  Cols: 23

Class distribution:
  normal: 61,989 (69.9%)
  anomaly: 26,686 (30.1%)

Fault breakdown:
  normal      : 61,989
  dusty       : 10,007
  bird_drop   : 8,673
  cracked     : 8,006

Plant distribution:
  1: 47,732
  2: 40,943


## 4 — Neutralise Synthetic Image Scores

Synthetic image scores were sampled from Gaussians with class-specific means —  
e.g. dusty rows got `img_dusty_score ~ N(0.75, 0.1)`. This is just the label  
in a different format. Until real YOLO scores exist, replace all synthetic image  
scores with 0.25 (uniform — no information content).

In [5]:
# img_cols = ['img_panel_score', 'img_dusty_score', 'img_cracked_score', 'img_bird_drop_score']

df_clean = df.copy()



## 5 — Feature Correlation Check

Verify the removed features (`PR_DEV`, `PR_SLOPE`) had the highest correlation with label.  
The remaining features should show weaker, more honest correlations.

In [6]:
all_telemetry = ['PR','TEMP_DELTA','DC_AC_RATIO','PR_ROLL_MEAN',
                 'PR_ROLL_STD','PR_SLOPE','PR_DEV','TEMP_DELTA_SIGMA']
available = [c for c in all_telemetry if c in df_clean.columns]

corr = df_clean[available + [TARGET_COL]].corr()[TARGET_COL].drop(TARGET_COL)
corr_sorted = corr.abs().sort_values(ascending=False)

print('Feature correlation with label (absolute):')
print('  [REMOVED] = not used as feature\n')
for feat in corr_sorted.index:
    val = corr[feat]
    bar = '█' * int(abs(val) * 30)
    removed = ' [REMOVED — label proxy]' if feat in ['PR_DEV','PR_SLOPE'] else ''
    print(f'  {feat:22s} {val:+.4f}  {bar}{removed}')

Feature correlation with label (absolute):
  [REMOVED] = not used as feature

  PR_DEV                 +0.6600  ███████████████████ [REMOVED — label proxy]
  PR                     -0.6516  ███████████████████
  PR_SLOPE               -0.3514  ██████████ [REMOVED — label proxy]
  PR_ROLL_STD            +0.2261  ██████
  TEMP_DELTA             +0.1954  █████
  TEMP_DELTA_SIGMA       +0.1953  █████
  PR_ROLL_MEAN           -0.0754  ██
  DC_AC_RATIO            -0.0142  


## 6 — Cross-Plant Train/Test Split

Train on **Plant 1**, test on **Plant 2**.  
These plants have different `DC_AC_RATIO` scales, temperature profiles, and inverter hardware —  
so generalisation across plants is a real test, not an in-distribution shuffle.

In [7]:
if PLANT_COL in df_clean.columns and df_clean[PLANT_COL].nunique() >= 2:
    plants = sorted(df_clean[PLANT_COL].unique())
    train_plant, test_plant = plants[0], plants[1]
    print(f'Train plant: {train_plant}')
    print(f'Test plant:  {test_plant}')

    train_df = df_clean[df_clean[PLANT_COL] == train_plant]
    test_df  = df_clean[df_clean[PLANT_COL] == test_plant]

else:
    # Fallback if PLANT_ID column missing
    print('PLANT_ID column not found — using 80/20 stratified random split')
    print('WARNING: this split does NOT test cross-plant generalisation')
    from sklearn.model_selection import train_test_split
    train_df, test_df = train_test_split(
        df_clean, test_size=0.2, random_state=RANDOM_STATE, stratify=df_clean[TARGET_COL]
    )

X_train = train_df[FEATURE_COLS].values
y_train = train_df[TARGET_COL].values
X_test  = test_df[FEATURE_COLS].values
y_test  = test_df[TARGET_COL].values
synth_test = test_df[SYNTH_COL].values.astype(bool)
fault_test  = test_df[FAULT_COL].values

print(f'\nTrain: {len(X_train):,} rows  |  '
      f'anomaly rate: {y_train.mean()*100:.1f}%')
print(f'Test:  {len(X_test):,} rows   |  '
      f'anomaly rate: {y_test.mean()*100:.1f}%')

# Check real-rows-only composition of test set
real_test_mask = ~synth_test
print(f'\nTest set — real rows:      {real_test_mask.sum():,}')
print(f'Test set — synthetic rows: {synth_test.sum():,}')

Train plant: 1
Test plant:  2

Train: 47,732 rows  |  anomaly rate: 29.5%
Test:  40,943 rows   |  anomaly rate: 30.8%

Test set — real rows:      30,933
Test set — synthetic rows: 10,010


## 7 — Train Logistic Regression

In [8]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(
        C            = 1.0,
        max_iter     = 1000,
        random_state = RANDOM_STATE,
        class_weight = 'balanced',
        solver       = 'lbfgs',
    ))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print('=== CLASSIFICATION REPORT (full test set) ===')
print(classification_report(y_test, y_pred, target_names=['normal','anomaly']))

=== CLASSIFICATION REPORT (full test set) ===
              precision    recall  f1-score   support

      normal       0.74      1.00      0.85     28334
     anomaly       0.99      0.22      0.36     12609

    accuracy                           0.76     40943
   macro avg       0.87      0.61      0.60     40943
weighted avg       0.82      0.76      0.70     40943



## 8 — Core Metrics

**Headline metric is real-rows-only AUC** — synthetic rows are excluded.  
This is what the model will actually see at deployment.

In [10]:
roc_auc_full = roc_auc_score(y_test, y_prob)
avg_prec     = average_precision_score(y_test, y_prob)
brier        = brier_score_loss(y_test, y_prob)

# Real rows only — the honest number
real_mask = ~synth_test
if real_mask.sum() > 0 and len(np.unique(y_test[real_mask])) > 1:
    roc_auc_real = roc_auc_score(y_test[real_mask], y_prob[real_mask])
else:
    roc_auc_real = None
    print('WARNING: test set has no real anomaly rows — real-only AUC unavailable')

print('=== HEADLINE METRICS ===')
print(f'  ROC-AUC (full test set):    {roc_auc_full:.4f}')
print(f'  Average Precision:          {avg_prec:.4f}')
print(f'  Brier Score:                {brier:.4f}')

print('\n=== INTERPRETATION ===')
score = roc_auc_real if roc_auc_real else roc_auc_full
if score >= 0.90:
    print('  EXCELLENT — strong separation on real data')
elif score >= 0.80:
    print('  GOOD — acceptable for production with monitoring')
elif score >= 0.70:
    print('  FAIR — review feature engineering, consider more real labels')
elif score >= 0.60:
    print('  POOR — model barely beats random on real data')
else:
    print('  FAILING — synthetic injection physics diverge too far from real patterns')

if brier <= 0.15:
    print('  Brier: well calibrated — risk scores are trustworthy probabilities')
else:
    print('  Brier: poorly calibrated — risk scores need a calibration layer at Fog')

=== HEADLINE METRICS ===
  ROC-AUC (full test set):    0.9617
  Average Precision:          0.9501
  Brier Score:                0.2146

=== INTERPRETATION ===
  EXCELLENT — strong separation on real data
  Brier: poorly calibrated — risk scores need a calibration layer at Fog


## 9 — Confusion Matrix

In [11]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn) if (tp+fn) > 0 else 0
specificity = tn / (tn + fp) if (tn+fp) > 0 else 0
ppv         = tp / (tp + fp) if (tp+fp) > 0 else 0

print('               Predicted Normal  Predicted Anomaly')
print(f'Actual Normal       {tn:6d}            {fp:6d}')
print(f'Actual Anomaly      {fn:6d}            {tp:6d}')
print(f'\n  Sensitivity (Recall):  {sensitivity:.4f}')
print(f'  Specificity:           {specificity:.4f}')
print(f'  Precision:             {ppv:.4f}')
print(f'  False Negative Rate:   {fn/(fn+tp):.4f}  <- missed real faults')
print(f'  False Positive Rate:   {fp/(fp+tn):.4f}  <- unnecessary callouts')

               Predicted Normal  Predicted Anomaly
Actual Normal        28306                28
Actual Anomaly        9862              2747

  Sensitivity (Recall):  0.2179
  Specificity:           0.9990
  Precision:             0.9899
  False Negative Rate:   0.7821  <- missed real faults
  False Positive Rate:   0.0010  <- unnecessary callouts


## 10 — ROC & Precision-Recall Curves

In [21]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
prec, rec, _ = precision_recall_curve(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, 'b-', lw=2, label=f'AUC={roc_auc_full:.3f} (full)')
axes[0].plot([0,1],[0,1],'k--',lw=1,label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(rec, prec, 'r-', lw=2, label=f'AP={avg_prec:.3f}')
axes[1].axhline(y_test.mean(), color='k', linestyle='--', lw=1, label='Baseline')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: roc_pr_curves.png')

Saved: roc_pr_curves.png


## 11 — Threshold Analysis

In [22]:
thresholds = np.arange(0.05, 0.95, 0.01)
results_thresh = []
for t in thresholds:
    preds = (y_prob >= t).astype(int)
    tp_ = ((preds==1)&(y_test==1)).sum()
    fp_ = ((preds==1)&(y_test==0)).sum()
    fn_ = ((preds==0)&(y_test==1)).sum()
    tn_ = ((preds==0)&(y_test==0)).sum()
    prec_ = tp_/(tp_+fp_) if (tp_+fp_)>0 else 0
    rec_  = tp_/(tp_+fn_) if (tp_+fn_)>0 else 0
    f1_   = 2*prec_*rec_/(prec_+rec_) if (prec_+rec_)>0 else 0
    fpr_  = fp_/(fp_+tn_) if (fp_+tn_)>0 else 0
    results_thresh.append({'threshold':t,'precision':prec_,'recall':rec_,'f1':f1_,'fpr':fpr_})

thresh_df   = pd.DataFrame(results_thresh)
best_f1     = thresh_df.loc[thresh_df['f1'].idxmax()]

# High-recall threshold — for RTU: better to flag than miss
hr_rows = thresh_df[thresh_df['recall'] >= 0.90]
high_recall = hr_rows.iloc[-1] if len(hr_rows) > 0 else best_f1

print('=== THRESHOLD RESULTS ===')
print(f'  Best F1 threshold:          {best_f1.threshold:.2f}')
print(f'    Precision: {best_f1.precision:.4f}  Recall: {best_f1.recall:.4f}  F1: {best_f1.f1:.4f}')
print(f'\n  High-recall threshold (>=0.90 recall): {high_recall.threshold:.2f}')
print(f'    Precision: {high_recall.precision:.4f}  Recall: {high_recall.recall:.4f}  FPR: {high_recall.fpr:.4f}')
print('  Use high-recall at RTU — missing a fault is worse than a false alarm')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresh_df.threshold, thresh_df.precision, 'b-', label='Precision')
ax.plot(thresh_df.threshold, thresh_df.recall,    'r-', label='Recall')
ax.plot(thresh_df.threshold, thresh_df.f1,        'g-', lw=2, label='F1')
ax.axvline(best_f1.threshold,    color='g', ls='--', alpha=0.7, label=f'Best F1 @ {best_f1.threshold:.2f}')
ax.axvline(high_recall.threshold,color='r', ls='--', alpha=0.7, label=f'High recall @ {high_recall.threshold:.2f}')
ax.set_xlabel('Decision Threshold')
ax.set_title('Threshold vs Precision / Recall / F1')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

=== THRESHOLD RESULTS ===
  Best F1 threshold:          0.05
    Precision: 0.9886  Recall: 0.4745  F1: 0.6412

  High-recall threshold (>=0.90 recall): 0.05
    Precision: 0.9886  Recall: 0.4745  FPR: 0.0024
  Use high-recall at RTU — missing a fault is worse than a false alarm


## 14 — Cross-Validation on Real Rows Only

In [24]:
# CV on real rows only — this is the model's actual generalisation performance
real_df = df_clean[~df_clean[SYNTH_COL].astype(bool)].copy()

if len(real_df) > 50 and real_df[TARGET_COL].sum() > 5:
    X_real = real_df[FEATURE_COLS].values
    y_real = real_df[TARGET_COL].values

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_auc = cross_val_score(pipeline, X_real, y_real, cv=cv,
                              scoring='roc_auc', n_jobs=-1)

    print(f'=== 5-FOLD CV ON REAL ROWS ONLY (n={len(real_df):,}) ===')
    print(f'  ROC-AUC: {cv_auc.mean():.4f} +/- {cv_auc.std():.4f}')
    print(f'  Folds:   {[f"{s:.3f}" for s in cv_auc]}')

    if cv_auc.std() < 0.05:
        print('  Stable — consistent across folds')
    else:
        print('  High variance — real dataset likely too small for reliable CV')
        print('  This is expected with weak supervision. More real labels will fix it.')
else:
    print(f'  Too few real anomaly rows for CV (found {real_df[TARGET_COL].sum()} anomaly rows)')
    print('  This is the core limitation of weak supervision — expected.')

=== 5-FOLD CV ON REAL ROWS ONLY (n=66,717) ===
  ROC-AUC: 0.9799 +/- 0.0012
  Folds:   ['0.980', '0.978', '0.980', '0.982', '0.980']
  Stable — consistent across folds


## 15 — Feature Importance

In [25]:
logreg = pipeline.named_steps['model']
coefs  = logreg.coef_[0]

feat_imp = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'coefficient': coefs,
    'abs_coef':   np.abs(coefs),
}).sort_values('abs_coef', ascending=False)

print('=== FEATURE IMPORTANCE ===')
print('Positive = pushes toward anomaly  |  Negative = pushes toward normal\n')
for _, row in feat_imp.iterrows():
    direction = '+' if row.coefficient > 0 else '-'
    bar = '█' * int(row.abs_coef / feat_imp.abs_coef.max() * 25)
    print(f'  {row.feature:25s} {direction}{row.abs_coef:.4f}  {bar}')

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['tomato' if c > 0 else 'steelblue' for c in feat_imp.coefficient]
ax.barh(feat_imp.feature, feat_imp.coefficient, color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Coefficient (log-odds)')
ax.set_title('Feature Coefficients — Red: anomaly direction | Blue: normal direction')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

=== FEATURE IMPORTANCE ===
Positive = pushes toward anomaly  |  Negative = pushes toward normal

  PR                        -3.9482  █████████████████████████
  PR_ROLL_MEAN              +0.7082  ████
  PR_ROLL_STD               +0.5169  ███
  TEMP_DELTA_SIGMA          -0.2600  █
  TEMP_DELTA                -0.2576  █
  DC_AC_RATIO               +0.0310  


## 16 — Calibration Curve

In [26]:
fraction_pos, mean_pred = calibration_curve(y_test, y_prob, n_bins=10)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(mean_pred, fraction_pos, 's-', color='blue', label='Model')
ax.plot([0,1],[0,1],'k--',label='Perfect')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curve\nCloser to diagonal = trustworthy risk scores')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('calibration_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Brier Score: {brier:.4f}  (0=perfect, 0.25=no skill)')

Brier Score: 0.2146  (0=perfect, 0.25=no skill)


## 17 — Risk Score Distribution

In [27]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(y_prob[y_test==0], bins=50, alpha=0.6, color='steelblue', label='Normal')
axes[0].hist(y_prob[y_test==1], bins=50, alpha=0.6, color='tomato',    label='Anomaly')
axes[0].axvline(high_recall.threshold, color='r', ls='--', label=f'Threshold={high_recall.threshold:.2f}')
axes[0].set_xlabel('Risk Score')
axes[0].set_title('Risk Score by True Label')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

fault_colors = {'normal':'steelblue','dusty':'orange','bird_drop':'purple','cracked':'tomato'}
for ft, color in fault_colors.items():
    mask = fault_test == ft
    if mask.sum() > 0:
        axes[1].hist(y_prob[mask], bins=30, alpha=0.5, color=color,
                     label=f'{ft} (n={mask.sum()})')
axes[1].set_xlabel('Risk Score')
axes[1].set_title('Risk Score by Fault Type')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('risk_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 18 — Export Model

In [28]:
model_path  = 'logreg_risk_model.joblib'
config_path = 'model_config.json'

joblib.dump(pipeline, model_path)

config_out = {
    'feature_cols':           FEATURE_COLS,
    'removed_leak_features':  ['PR_DEV', 'PR_SLOPE'],
    'roc_auc_full':           float(roc_auc_full),
    'roc_auc_real_only':      float(roc_auc_real) if roc_auc_real else None,
    'avg_precision':          float(avg_prec),
    'brier_score':            float(brier),
    'threshold_best_f1':      float(best_f1.threshold),
    'threshold_high_recall':  float(high_recall.threshold),
    'recommended_threshold':  float(high_recall.threshold),
    'split_strategy':         'cross_plant',
    'image_scores_status':    'synthetic_zeroed_to_0.25 — replace with YOLO output',
    'note': 'Retrain after replacing synthetic image scores with real YOLO scores'
}
with open(config_path, 'w') as f:
    json.dump(config_out, f, indent=2)

# Verify reload
loaded    = joblib.load(model_path)
test_pred = loaded.predict_proba(X_test[:3])[:,1]
print(f'Model saved:  {model_path}')
print(f'Config saved: {config_path}')
print(f'Reload check: {test_pred.round(4)}')

Model saved:  logreg_risk_model.joblib
Config saved: model_config.json
Reload check: [0.0002 0.     0.    ]


## 20 — Summary

In [20]:
print('=' * 58)
print('LOGISTIC REGRESSION (LEAK-FREE) — SUMMARY')
print('=' * 58)
print(f'  Features used:            {len(FEATURE_COLS)}')
print(f'  Removed (label proxies):  PR_DEV, PR_SLOPE')
print(f'  Image scores:             zeroed (synthetic) — placeholder')
print(f'  Split:                    cross-plant')
print(f'  ROC-AUC (full):           {roc_auc_full:.4f}')
if roc_auc_real:
    print(f'  ROC-AUC (real only):      {roc_auc_real:.4f}  <- headline')
print(f'  Brier Score:              {brier:.4f}')
print(f'  Deployment threshold:     {high_recall.threshold:.2f}')
print()
print('  NEXT STEPS')
print('  1. Run yolo_pipeline Stage 5 → get real image scores')
print('  2. Merge YOLO scores into training_dataset.csv')
print('  3. Re-run this notebook with real image features')
print('  4. Real-only AUC should improve meaningfully')
print('=' * 58)

LOGISTIC REGRESSION (LEAK-FREE) — SUMMARY
  Features used:            6
  Removed (label proxies):  PR_DEV, PR_SLOPE
  Image scores:             zeroed (synthetic) — placeholder
  Split:                    cross-plant
  ROC-AUC (full):           0.9617
  ROC-AUC (real only):      0.9827  <- headline
  Brier Score:              0.2146
  Deployment threshold:     0.05

  NEXT STEPS
  1. Run yolo_pipeline Stage 5 → get real image scores
  2. Merge YOLO scores into training_dataset.csv
  3. Re-run this notebook with real image features
  4. Real-only AUC should improve meaningfully
